In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
print(assistant.rag('How do I run Ollama locally?'))

To run Ollama locally, follow these steps:

1. **Install Ollama**: Visit [https://ollama.com/download](https://ollama.com/download) and choose your operating system:
   - **macOS**: Download the `.pkg` and install it.
   - **Windows**: Download the `.msi` and install it.
   - **Linux**: Run the command `curl -fsSL https://ollama.com/install.sh | sh` in the terminal.

2. **Run Ollama**: Open a terminal and type `ollama run llama3`. This will download the LLaMA 3 model (~4GB), start the model locally, and open a chat-like interface where you can type questions.

3. **Test the Ollama local server**: Run the command `curl http://localhost:11434` to test the server. You should receive a response similar to `{"models": [...]}`.

4. **Install the Python client**: Install the Python client with `pip install ollama`.

5. **Run a minimal Python example**: Use the following Python code to test Ollama:
   ```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "

In [8]:
print(assistant.rag("How do I run Olama locally?"))

There is no information provided about running Olama locally. The given context does not contain any details or instructions on how to run Olama. If you need assistance with running Olama, please provide more context or information about Olama and its requirements.


In [9]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=messages,
)

response.output_text

"You'd like to join a course, however, I need a bit more information. Could you please provide more details about the course you're interested in, such as its name, subject, or platform it's being hosted on? That way, I can give you a more accurate answer about whether it's still possible to join."

In [10]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )
    

In [13]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'required': ["query"],
        'additionalProperties': False
    }
}

In [15]:
response  =openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=messages,
    tools=[search_tool]
)

In [16]:
len(response.output)

2

In [20]:
call = response.output[1]

In [21]:
call

ResponseFunctionToolCall(arguments='{"query":"joining the course deadline"}', call_id='skdj08y5h', name='search', type='function_call', id='skdj08y5h', namespace=None, status='completed')

In [22]:
import json

args = json.loads(call.arguments)
args

{'query': 'joining the course deadline'}

In [23]:
call.name

'search'

In [24]:
results = search(**args)

In [26]:
result_json = json.dumps(results, indent=2)

In [27]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}


In [28]:
messages.append(call)

In [29]:
messages.append(function_call_output)

In [30]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"joining the course deadline"}', call_id='skdj08y5h', name='search', type='function_call', id='skdj08y5h', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'skdj08y5h',
  'output': '[\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2025."\n  },\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "q

In [32]:
response = openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=messages,
    tools=[search_tool]
)

In [33]:
print(response.output_text)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [34]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(819, 33)

In [37]:
def calculate_llama_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
    "input_cost": input_cost,
    "output_cost": output_cost,
    "total_cost": total_cost
    }

result = calculate_llama_price(819, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.00014265


In [39]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
    }
    

In [40]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords.
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]


In [41]:
response = openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=messages,
    tools=[search_tool]
)


In [42]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        pritn(item.content[0].text)

function_call: search {"query":"joining the course"}


In [43]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function.\nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords.\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01kv5f9pkden1tg5fr70pcap7a', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"joining the course"}', call_id='51fv3vbzs', name='search', type='function_call', id='51fv3vbzs', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': '51fv3vbzs',
  'output': '[\n  {\n 

In [48]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model ='llama-3.3-70b-versatile',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"joining the course"}
iteration #2...
ASSISTANT:
Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [96]:
def agent_loop(instructions, question, model='llama-3.3-70b-versatile'):
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1
    last_answer = None
    last_response = None
    
    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        last_response = response
        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return {
        'answer': last_answer,
        'response': last_response,
        'tokens': {
            'input': last_response.usage.input_tokens,
            'output': last_response.usage.output_tokens,
        }
    }

In [98]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords fro the user question as possible when making first requests.

Make multiple searches. First perform search, analyse the results
and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

In [99]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"joining the course"}
iteration #2...
ASSISTANT:
Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [100]:
result

{'answer': 'Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'response': Response(id='resp_01kv5n6q3xfc9a6a6kwbd5f7bm', created_at=1781527698.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='llama-3.3-70b-versatile', object='response', output=[ResponseReasoningItem(id='resp_01kv5n6q3xfc9tq52pn2g5jxb4', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'), ResponseOutputMessage(id='msg_01kv5n6q3xfca93enr2pfdv2qp', content=[ResponseOutputText(annotations=[], text='Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.', type='output_text', logprobs=None)], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameter

In [101]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"Queen Gambit"}
iteration #2...
ASSISTANT:
The Queen's Gambit is a popular chess opening that starts with the moves 1.d4 d5 2.c4. It is considered one of the oldest and most aggressive openings in chess, and is often played by players of all skill levels.

The Queen's Gambit is characterized by the white player offering a pawn to the black player, which can be accepted or declined. If the black player accepts the pawn, the white player will often launch a powerful attack on the black position, while if the black player declines the pawn, the white player will focus on controlling the center of the board and developing their pieces.

The Queen's Gambit has been played by many famous chess players throughout history, including world champions such as Bobby Fischer and Garry Kasparov. It is considered a challenging and complex opening, and is often used by players who want to take the initiative and play aggressively.

In addition to its use 

In [ ]:
# Can't able to do the below part cause of compatibility issue i'm using GROQ API

In [2]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [3]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

NameError: name 'search' is not defined

In [104]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [105]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [106]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [107]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [108]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=openai_client

)

In [86]:
# result = runner.loop(
#     prompt="How do I run Olama locally?",
#     callback=callback,
# )

AttributeError: 'OpenAI' object has no attribute 'send_request'

In [115]:
result = agent_loop(instructions, "How do I run Olama locally?")

iteration #1...


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search {"query": "running Olama locally"} </function>'}}

In [114]:
result.cost

AttributeError: 'dict' object has no attribute 'cost'

In [117]:
result = agent_loop(instructions, question)
print(result['answer'])
print(result['tokens'])
print(result['response'].usage)

iteration #1...
function_call: search {"query":"Queen Gambit"}
iteration #2...
ASSISTANT:
The Queen's Gambit is a popular chess opening that starts with the moves 1.d4 d5 2.c4. It is a versatile and aggressive opening that offers a range of possibilities for counterplay and complexity. The Queen's Gambit is a favorite among many chess players, from beginners to Grandmasters, and has been used in countless games throughout history. 

In recent years, the term "Queen's Gambit" has also become associated with a Netflix series of the same name, which follows the story of a young orphan girl who becomes a chess prodigy and competes in high-stakes tournaments. The show explores themes of obsession, addiction, and self-discovery, and features a blend of drama, suspense, and chess strategy.
The Queen's Gambit is a popular chess opening that starts with the moves 1.d4 d5 2.c4. It is a versatile and aggressive opening that offers a range of possibilities for counterplay and complexity. The Queen

In [122]:
result.all_messages

AttributeError: 'dict' object has no attribute 'all_messages'

In [124]:
result2 = agent_loop(instructions, "How do I run a different model?")

iteration #1...
function_call: search {"query":"run a different model"}
iteration #2...
ASSISTANT:
To run a different model, you need to specify the model name when creating a client or making a request. For example, to use the "gemini-2.0-flash" model, you can use the following code:

```python
import google.generativeai as genai
model = genai.GenerativeModel('gemini-2.0-flash')
```

Alternatively, you can use the `client.chat.completions.create` method and specify the model name in the `model` parameter:

```python
response = client.chat.completions.create(
    model="gemini-2.0-flash",
    messages=[{"role": "user", "content": prompt}],
    tools=tools_schema,  # may need adapting per provider
)
```

Make sure to replace "gemini-2.0-flash" with the actual name of the model you want to use. Also, note that different providers may have different model names and requirements, so be sure to check the documentation for the specific provider you are using.


In [1]:
result2 = agent_loop(instructions, "Can I still join the course after the start date?")

NameError: name 'agent_loop' is not defined